## Setup

In [ ]:
# HF token
from huggingface_hub import login
hf_token = "xxx"
login(hf_token)
# NDIF token
from nnsight import CONFIG
CONFIG.set_default_api_key("xxx")

d:\2026\LensHub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`src.jlens_fromhf.from_hf` rewrites `jlens.hf.from_hf` to wrap an nnsight model with the same interfaces.

In [2]:
from nnsight import LanguageModel
import jlens
jlens.configure_logging()
from src.jlens_fromhf import from_hf
MODEL_NAME = "meta-llama/Llama-3.1-8b"
nn_model = LanguageModel(MODEL_NAME, remote=True)
tokenizer = nn_model.tokenizer
model = from_hf(nn_model, force_bos=False)

Verify the model is able to run remotely.

In [3]:
prompt = "Fact: The number of legs on the animal that spins webs is"
with nn_model.generate(prompt, max_new_tokens=3, remote=True) as tracer:
    tokens = list().save()
    for step in tracer.iter[:]:
        tokens.append(nn_model.lm_head.output[0, -1].argmax(dim=-1))

print([tokenizer.decode(t) for i, t in enumerate(tokens)])

⬇ Downloading: 100%|██████████| 632/632 [00:00<00:00]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[' eight', '8', '.\n']


In [4]:
prompt = "Fact: The currency used in the country shaped like a boot is"
with nn_model.generate(prompt, max_new_tokens=3, remote=True) as tracer:
    tokens = list().save()
    for step in tracer.iter[:]:
        tokens.append(nn_model.lm_head.output[0, -1].argmax(dim=-1))

print([tokenizer.decode(t) for i, t in enumerate(tokens)])

⬇ Downloading: 100%|██████████| 637/637 [00:00<00:00]

[' the', ' Euro', ' dollar']


In [5]:
prompt = "iac^ilege^iac\nptest^yi^ptest\nks^ixe^"    # the ABA rule
with nn_model.generate(prompt, max_new_tokens=3, remote=True) as tracer:
    tokens = list().save()
    for step in tracer.iter[:]:
        tokens.append(nn_model.lm_head.output[0, -1].argmax(dim=-1))

print([tokenizer.decode(t) for i, t in enumerate(tokens)])

⬇ Downloading: 100%|██████████| 633/633 [00:00<00:00]

['ks', '\n', '^']


## JLens in `nnsight`

`src.jlens_fitting.fit_jlens` rewrites `jlens.fitting.fit` to run remotely on NDIF.

In [6]:
from jlens.examples import load_wikitext_prompts
from src.jlens_fitting import fit_jlens

# prompts = load_wikitext_prompts(n_prompts=100)
prompts = ["Fact: The number of legs on the animal that spins webs is"]
lens = fit_jlens(
    model, prompts, dim_batch=100, max_seq_len=128, checkpoint_path="ckpt.pt", skip_first=2
)
lens.save("jacobian_lens.pt")

[    17s + 17.01s] fit: n_layers=32 d_model=4096, fitting 31 source layers (target=L31) on 1 prompts
[    18s +  1.52s]   resuming from checkpoint: 1/1 prompts processed
[    23s +  4.87s] fit: done, 1 prompts


Load a Jlens file trained with 100 promppts and default `skip_first`.

In [7]:
import jlens
lens = jlens.JacobianLens.from_pretrained(
    name_or_path="./lens/jlens", filename="jacobian_lens_llama_31_8b_prompts_100_seq_128.pt"
)

`src.jlens_tools.apply_jlens` rewrites `lens.apply`.

In [8]:
prompt = "Fact: The number of legs on the animal that spins webs is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

from src.jlens_tools import apply_jlens
jlens_logits, model_logits, _ = apply_jlens(lens, model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = apply_jlens(lens, model, prompt, layers=layers, positions=[-2], use_jacobian=False)

⬇ Downloading: 100%|██████████| 447k/447k [00:00<00:00]


Processing layer 8...


⬇ Downloading: 100%|██████████| 201k/201k [00:00<00:00]


Processing layer 16...


⬇ Downloading: 100%|██████████| 196k/196k [00:00<00:00]


...

Processing layer 24...


⬇ Downloading: 100%|██████████| 201k/201k [00:00<00:00]


Processing layer 30...


⬇ Downloading: 100%|██████████| 196k/196k [00:00<00:00]


⬇ Downloading: 100%|██████████| 189k/189k [00:00<00:00]


Print Jlens readouts as in the official repo.

In [9]:
def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]

for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  8 logit-lens: ['uyo', 'omen', 'pread', ' intact', 'fect']
L  8 J-lens:     [' ..', ' ..\n', ' ...', ' ..\n\n', 's']
L 16 logit-lens: ['��', 'iew', 'owan', ' industri', 'Forms']
L 16 J-lens:     ['?\n', '.\n', '...\n', '.', '\n']
L 24 logit-lens: [' spiders', ' webs', ' spider', ' spun', ' Spinner']
L 24 J-lens:     [' spiders', ' spider', ' Spider', ' webs', ' is']
L 30 logit-lens: [' to', ' is', '.\n', ' in', '?\n']
L 30 J-lens:     ['.\n', ' is', '?\n', '.', ' to']
model:           [' is', ' to', '.\n', ' in', '.']


`src.jlens_tools.compute_slice_lens` rewrites `jlens.vis.compute_slice`. `slice_data` will be reused for both inline and served page.

In [10]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt

from src.jlens_tools import compute_slice_lens

# English gloss for Qwen's Chinese/Japanese/Korean vocab tokens (machine-
# generated, best-effort), shown next to
# the token in the page (alt_token=).
gloss = {
    int(k): v for k, v in json.load(gzip.open("jacobian-lens/assets/qwen_gloss.json.gz")).items()
}

example = next(e for e in EXAMPLES if e.slug == "multihop")
prompt = resolve_prompt(example, tokenizer)
print("prompt: ", prompt)
slice_data = compute_slice_lens(
    model,
    lens,
    prompt,
    layer_stride=1,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
)

prompt:  Fact: The capital of Japan is Tokyo.
Fact: The currency used in the country shaped like a boot is


⬇ Downloading: 100%|██████████| 4.70M/4.70M [00:00<00:00]


Processing layer 0...


⬇ Downloading: 100%|██████████| 4.60M/4.60M [00:00<00:00]


Processing layer 1...


...

⬇ Downloading: 100%|██████████| 4.58M/4.58M [00:00<00:00]


⬇ Downloading: 100%|██████████| 4.57M/4.57M [00:00<00:00]


⬇ Downloading: 100%|██████████| 4.56M/4.56M [00:00<00:00]


⬇ Downloading: 100%|██████████| 4.51M/4.51M [00:00<00:00]


⬇ Downloading: 100%|██████████| 4.38M/4.38M [00:00<00:00]


In [11]:
from jlens.vis import build_page, notebook_iframe
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
)
notebook_iframe(page)

In [12]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

# example = next(e for e in EXAMPLES if e.slug == "ascii-face")
# prompt = resolve_prompt(example, tokenizer)
# prompt = "Fact: The number of legs on the animal that spins webs is"

# slice_data = compute_slice_lens(model, lens, prompt, mask_display=True)
out_dir = Path("slices") / example.slug
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath("slices"))
    _jlens_httpd = HTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()
print(f"-> http://localhost:{_jlens_httpd.server_address[1]}/{example.slug}/")

-> http://localhost:10789/multihop/


127.0.0.1 - - [14/Sep/2026 13:51:33] "GET /multihop/ HTTP/1.1" 200 -
127.0.0.1 - - [14/Sep/2026 13:51:33] "GET /multihop/meta.json HTTP/1.1" 200 -
127.0.0.1 - - [14/Sep/2026 13:51:33] "GET /multihop/slice.bin HTTP/1.1" 200 -
127.0.0.1 - - [14/Sep/2026 13:51:33] code 404, message File not found
127.0.0.1 - - [14/Sep/2026 13:51:33] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [14/Sep/2026 13:51:33] code 404, message File not found
127.0.0.1 - - [14/Sep/2026 13:51:33] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [14/Sep/2026 13:51:37] "GET /multihop/ranks/27367.bin HTTP/1.1" 200 -
127.0.0.1 - - [14/Sep/2026 13:51:38] "GET /multihop/ranks/574.bin HTTP/1.1" 200 -


Stop the server as necessary.

In [13]:
# 1. Stop the background daemon thread
_jlens_httpd.shutdown()
# 2. Close the socket and free the local port
_jlens_httpd.server_close()
# 3. Remove the global variable
del _jlens_httpd
print("Server stopped successfully.")

Server stopped successfully.


`src.jlens_headwise.jlens_head_by_mul` implements the readouts from weight matrices $W_Q$, $W_K$, $W_V$ and $W_O$ in a specified attention head with $W_UJW_{\{Q,K,V,O\}}$ as described in the caption of Figure 93. The selected head (layer 16, head 3) is identified as a retrieval head from the reproduced CMA results of _Emergent Symbolic Mechanisms Support Abstract Reasoning in Large Language Models_ ([link](https://arxiv.org/pdf/2502.20332)).

![CMA scores for Retrieval Head](assets/Score_Patching_ABA_to_ABA_heatmap.png)

In [14]:
from src.jlens_headwise import jlens_head_by_mul
layer = 16
head = 3
token_scores = jlens_head_by_mul(nn_model, lens, layer, head)

d_model=4096 n_heads=32 n_kv_heads=8
head_dim=128 kv_groups=4 kv_h=0


⬇ Downloading: 100%|██████████| 820k/820k [00:00<00:00]


⬇ Downloading: 100%|██████████| 433k/433k [00:00<00:00]


W_Q_scores.shape=torch.Size([128256])


⬇ Downloading: 100%|██████████| 821k/821k [00:00<00:00]


...

⬇ Downloading: 100%|██████████| 447k/447k [00:00<00:00]


W_V_scores.shape=torch.Size([128256])


⬇ Downloading: 100%|██████████| 821k/821k [00:00<00:00]


⬇ Downloading: 100%|██████████| 440k/440k [00:00<00:00]

W_O^T_scores.shape=torch.Size([128256])


`from src.jlens_headwise.rank_tokens` reuses `jlens.vis._meaningful_token_mask` and `jlens.vis._ranks_of` to show the top-ranked word-like tokens with their ranks.

In [15]:
from src.jlens_headwise import rank_tokens
top_k = 6
rank_tokens(nn_model.tokenizer, token_scores, layer, head, top_k)


J-Lens Readout | Layer 16 | Head 3
W_Q (rank)                W_K (rank)                W_V (rank)                W_O^T (rank)             
likewise (59)             or (17)                   s (17)                    RP (1)                   
similarly (63)            TokenName (41)            in (47)                   RK (2)                   
TokenName (142)           either (61)               b (53)                    Rosenstein (3)           
https (143)               데이트 (73)                  ob (56)                   RX (5)                   
was (144)                 his (78)                  t (66)                    Dumbledore (6)           
equally (166)             Uygu (80)                 c (70)                    RF (7)                   


`src.jlens_headwise.jlens_head_by_fw` uses normal forward to get the final logits.

In [16]:
from src.jlens_headwise import jlens_head_by_fw
layer = 16
head = 3
token_scores = jlens_head_by_fw(nn_model, lens, layer, head)

d_model=4096 n_heads=32 n_kv_heads=8
head_dim=128 kv_groups=4 kv_h=0


⬇ Downloading: 100%|██████████| 820k/820k [00:00<00:00]


⬇ Downloading: 100%|██████████| 433k/433k [00:00<00:00]


W_Q_scores.shape=torch.Size([128256])


⬇ Downloading: 100%|██████████| 821k/821k [00:00<00:00]


...

⬇ Downloading: 100%|██████████| 434k/434k [00:00<00:00]


W_V_scores.shape=torch.Size([128256])


⬇ Downloading: 100%|██████████| 821k/821k [00:00<00:00]


⬇ Downloading: 100%|██████████| 437k/437k [00:00<00:00]

W_O^T_scores.shape=torch.Size([128256])


In [17]:
from src.jlens_headwise import rank_tokens
top_k = 6
rank_tokens(nn_model.tokenizer, token_scores, layer, head, top_k)


J-Lens Readout | Layer 16 | Head 3
W_Q (rank)                W_K (rank)                W_V (rank)                W_O^T (rank)             
likewise (29)             or (22)                   s (20)                    DH (0)                   
similarly (34)            either (49)               in (38)                   RP (1)                   
equally (100)             https (75)                b (45)                    Dumbledore (2)           
â (115)                   herself (86)              t (61)                    RK (3)                   
Â (117)                   http (99)                 c (65)                    D (4)                    
ones (130)                his (110)                 L (68)                    DK (5)                   


## Taylor Lens

The first-order Taylor approximation for a vector-valued, multivariate function $y=f(x)$ is $f(x)=f(a)+J(a)(x-a)+o(||x-a||_2)$. J-lens uses $f(x)=J_{\ell}x$ instead, where $x$ and $f(x)$ are the activations at a source layer $\ell$ and the final layer respectively, and $J_{\ell}=E_{t,t'\geq t,prompt}[\frac{\partial h_{final,t'}}{\partial h_{\ell,t}}]$. So is $J(a=0)=J_{\ell}$ and is $f(a=0)=0$? And is there a potentially better choice of $a$?

Fitting a Taylor Lens (talens) requires ablating each layer and each position separately, so it is #layers x #positions slower than fitting a Jacobian Lens. In my experiment, it spent >20h for a prompt with 31 layers and 11 valid positions.

In [18]:
# from src.talens_fitting import fit_talens
# talens = fit_talens(
#         model, prompts, dim_batch=10, max_seq_len=128, checkpoint_path=f"ckpt.pt", source_layers=[layer], pos_stride=1, skip_first=2, expand_at=0
#     )
# talens.save(f"taylor_lens.pt")

`src.talens.TaylorLens` wraps methods in the style of `jlens.JacobianLens`.

In [19]:
from src.talens import TaylorLens
talens = TaylorLens.from_pretrained(
    name_or_path="./lens/talens/spider_at_zero", filename="taylor_lens.pt"
)
talens_logits, _model_logits, _ = talens.apply(model, prompt, layers=layers, positions=[-2], expand_at=0)

⬇ Downloading: 100%|██████████| 733k/733k [00:00<00:00]


Processing layer 8...


⬇ Downloading: 100%|██████████| 198k/198k [00:00<00:00]


Processing layer 16...


⬇ Downloading: 100%|██████████| 201k/201k [00:00<00:00]


Processing layer 24...


⬇ Downloading: 100%|██████████| 196k/196k [00:00<00:00]


Processing layer 30...


⬇ Downloading: 100%|██████████| 200k/200k [00:00<00:00]


⬇ Downloading: 100%|██████████| 191k/191k [00:00<00:00]


In [20]:
import math
for layer in layers:
    print(f"L{layer:>3} ||f(0)||/sqrt(d): {talens.residuals[layer].norm().item()/math.sqrt(talens.residuals[layer].shape[0]):<20} "
          f"diff in error: {(jlens_logits[layer][0] - model_logits[0]).norm().item():>20} "
          f"J<->T {(talens_logits[layer][0] - model_logits[0]).norm().item()}")

L  8 ||f(0)||/sqrt(d): 0.6934938430786133   diff in error:   1219.2269287109375 J<->T 1296.877685546875
L 16 ||f(0)||/sqrt(d): 0.5687042474746704   diff in error:    769.2557373046875 J<->T 1309.3485107421875
L 24 ||f(0)||/sqrt(d): 0.550128698348999    diff in error:    743.9044799804688 J<->T 981.0711669921875
L 30 ||f(0)||/sqrt(d): 1.3088687658309937   diff in error:    433.6539306640625 J<->T 1232.029052734375


### Rethinking: Why does Taylor Lens perform worse?

Since $f(x)\approx f(a)+J(a)(x-a)=f(a)-J(a)a+J(a)x$, if $f(a)-J(a)a=0$, then we have $f(x)\approx J(a)x$. So we can estimate $J(a)$ by $J(a)=E[\frac{\partial f(a)}{\partial a}]$ over a large corpus of $a$, which is exactly what $J_{\ell}$ does!

## Linear Lens (Next Weeks)

In terms of minimizing the approximation error in $h_{final}=A_{\ell}h_{\ell}+b_{\ell}$, we can use the close-form solution for linear regression with MSE: $A_i^*=\argmin||H_{final,i}-AH_{\ell}||_2^2=(H_{\ell}^T H_{\ell})^{-1}H_{\ell}^T H_{final,i},i=1,\cdots,d_{model}$, where $H_{\ell}\in R^{d_{sample}\times d_{model}}$ and $H_{final,i}\in R^{d_{sample}}$ are the stacked $d_{sample}\geq d_{model}$ $h_{\ell}$'s and $h_{final,i}$'s respectively.

There are two papers similar to this. 

The first is _Jump to Conclusions: Short-Cutting Transformers with Linear Transformations_ ([link](https://arxiv.org/pdf/2303.09435)), which uses a matrix $A$ to map from a layer $\ell$ to another layer $\ell'$ but does not examine it from a lens view.

![Linear Map](./assets/linearmap.png)

The second paper is _Eliciting Latent Predictions from Transformers with the Tuned Lens_ ([link](https://arxiv.org/pdf/2303.08112)), which uses $A_{\ell}h_{\ell}+b_{\ell}$ to approximate $h_{final}$, but minimizes the KL.

![Tuned Lens A&b](./assets/tunedlens1.png)
![Tuned Lens KL](./assets/tunedlens2.png)

In practice, we can pad an additional $1$ to $h_{\ell}$ and absorb $b_{\ell}$ to $A_{i,\ell+1},i=1,\cdots,d_{model}$ to apply the close-form solution.